# Imports

In [2]:
import os
import pandas as pd
import numpy as np

from openai import OpenAI

# Import datasets

In [4]:
import re
from pathlib import Path

DATA_DIR = Path("../data")  
FILE_2025 = DATA_DIR / "flat_data_2025.csv"

def load_any(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported file type: {path.suffix}")

df = load_any(FILE_2025)

print(df.shape)
print(df.columns.tolist())
df.head()

(2013, 9)
['title', 'url', 'news_site', 'published_date', 'quarter_year', 'content', 'topic', 'sentiment_score', 'explanation']


,title,url,news_site,published_date,quarter_year,content,topic,sentiment_score,explanation
0,"19,600 BTO flats to launch this year, includin...",https://www.channelnewsasia.com/singapore/1960...,cna,2025-01-16 17:00:00,2025Q1,Flats with shorter waiting times will become “...,"BTO & SBF Supply, Launches, and Demand",0.8,The article discusses increased BTO supply and...
1,"19,600 BTO flats to launch this year, includin...",https://www.channelnewsasia.com/singapore/1960...,cna,2025-01-16 17:00:00,2025Q1,Flats with shorter waiting times will become “...,Public Housing Eligibility & Schemes,0.6,New schemes like Family Care Scheme and change...
2,"19,600 BTO flats to launch this year, includin...",https://www.channelnewsasia.com/singapore/1960...,cna,2025-01-16 17:00:00,2025Q1,Flats with shorter waiting times will become “...,Affordability & Cost of Living,0.3,The article mentions efforts to address housin...
3,"19,600 BTO flats to launch this year, includin...",https://www.channelnewsasia.com/singapore/1960...,cna,2025-01-16 17:00:00,2025Q1,Flats with shorter waiting times will become “...,"Urban Planning, Master Plans & New Town Develo...",0.4,"The 2025 Draft Masterplan is mentioned, focusi..."
4,"19,600 BTO flats to launch this year, includin...",https://www.channelnewsasia.com/singapore/1960...,cna,2025-01-16 17:00:00,2025Q1,Flats with shorter waiting times will become “...,Cooling Measures & Government Policies,0.2,The government's cautious approach to new cool...


# Normalise fields

In [5]:
title_col = "title"
topic_col = "topic"
score_col = "sentiment_score"
date_col = "published_date"
quarter_col = "quarter_year"

df["title_norm"] = df[title_col].astype(str).str.strip()

df["topic_norm"] = df[topic_col].astype(str).str.strip()

df["sentiment_score"] = pd.to_numeric(df[score_col], errors="coerce")

df["published_date"] = pd.to_datetime(df[date_col], errors="coerce")

# article year
df["year"] = df["published_date"].dt.year
if df["year"].isna().all() and quarter_col is not None:
    df["year"] = (
        df[quarter_col]
        .astype(str)
        .str.extract(r"(\d{4})")[0]
    )
    df["year"] = pd.to_numeric(df["year"], errors="coerce")

df["article_key"] = (
    df["title_norm"].fillna("")
    + " | "
    + df["published_date"].astype(str).fillna("")
    + " | "
    + df["topic_norm"].fillna("")
)

In [6]:
# Build Retrieval Text
df["retrieval_text"] = (
    df["title_norm"]
    + "\n"
    + df["topic_norm"]
    + "\n"
    + df["content"]
)

df["retrieval_text"].head(1)

0    19,600 BTO flats to launch this year, includin...
Name: retrieval_text, dtype: str

# Batch processing

In [7]:
def split_into_chunks(text, max_chars=3000, overlap_chars=300):
    """
    Split text into overlapping chunks.
    Uses paragraph boundaries first, then falls back to raw slicing if needed.
    """
    text = re.sub(r"\n{3,}", "\n\n", str(text).strip())
    if not text:
        return []

    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks = []
    current = ""

    def flush_current():
        nonlocal current
        if current.strip():
            chunks.append(current.strip())
        current = ""

    for para in paragraphs:
        # If a single paragraph is too large, split it directly
        if len(para) > max_chars:
            flush_current()
            start = 0
            while start < len(para):
                end = min(start + max_chars, len(para))
                chunks.append(para[start:end].strip())
                if end == len(para):
                    break
                start = max(0, end - overlap_chars)
            continue

        # Try to append paragraph to current chunk
        candidate = para if not current else current + "\n\n" + para
        if len(candidate) <= max_chars:
            current = candidate
        else:
            flush_current()
            current = para

    flush_current()

    # Add overlap between chunks
    if overlap_chars > 0 and len(chunks) > 1:
        overlapped = []
        for i, chunk in enumerate(chunks):
            if i == 0:
                overlapped.append(chunk)
            else:
                prev_tail = chunks[i - 1][-overlap_chars:]
                overlapped.append(prev_tail + "\n\n" + chunk)
        chunks = overlapped

    return chunks

In [10]:
chunk_rows = []

for _, row in df.iterrows():
    chunks = split_into_chunks(
        row["retrieval_text"],
        max_chars=3000,
        overlap_chars=300
    )

    for i, chunk in enumerate(chunks):
        chunk_rows.append({
            "article_key": row["article_key"],
            "chunk_id": i,
            "title": row["title"],
            "topic_norm": row["topic_norm"],
            "year": row["year"],
            "news_site": row["news_site"],
            "published_date": row["published_date"],
            "retrieval_text": chunk
        })

chunks_df = pd.DataFrame(chunk_rows)

print(chunks_df.shape)
chunks_df.head()

(5016, 8)


,article_key,chunk_id,title,topic_norm,year,news_site,published_date,retrieval_text
0,"19,600 BTO flats to launch this year, includin...",0,"19,600 BTO flats to launch this year, includin...","BTO & SBF Supply, Launches, and Demand",2025,cna,2025-01-16 17:00:00,"19,600 BTO flats to launch this year, includin..."
1,"19,600 BTO flats to launch this year, includin...",1,"19,600 BTO flats to launch this year, includin...","BTO & SBF Supply, Launches, and Demand",2025,cna,2025-01-16 17:00:00,"ability amid rising resale prices, the governm..."
2,"19,600 BTO flats to launch this year, includin...",0,"19,600 BTO flats to launch this year, includin...",Public Housing Eligibility & Schemes,2025,cna,2025-01-16 17:00:00,"19,600 BTO flats to launch this year, includin..."
3,"19,600 BTO flats to launch this year, includin...",1,"19,600 BTO flats to launch this year, includin...",Public Housing Eligibility & Schemes,2025,cna,2025-01-16 17:00:00,"lity amid rising resale prices, the government..."
4,"19,600 BTO flats to launch this year, includin...",0,"19,600 BTO flats to launch this year, includin...",Affordability & Cost of Living,2025,cna,2025-01-16 17:00:00,"19,600 BTO flats to launch this year, includin..."


In [12]:
def estimate_tokens(text):
    # Rough heuristic: 1 token ~= 4 characters
    return max(1, len(str(text)) // 4)

def make_batches(texts, max_batch_tokens=200000, max_batch_items=64):
    batches = []
    current_batch = []
    current_tokens = 0

    for text in texts:
        t = estimate_tokens(text)

        # If one text alone is huge, still send it by itself
        if t > max_batch_tokens:
            if current_batch:
                batches.append(current_batch)
                current_batch = []
                current_tokens = 0
            batches.append([text])
            continue

        if (
            current_batch
            and (
                current_tokens + t > max_batch_tokens
                or len(current_batch) >= max_batch_items
            )
        ):
            batches.append(current_batch)
            current_batch = [text]
            current_tokens = t
        else:
            current_batch.append(text)
            current_tokens += t

    if current_batch:
        batches.append(current_batch)

    return batches

# Embedding Model

In [13]:
# api key from platform.ai
api_key = 'sk-Rv-F9yu8jWjmjHI65Bjq2A'
base_url = 'https://api.ai.tech.gov.sg/platform/models'
model = "text-embedding-3-small"

client = OpenAI(
    api_key=api_key, 
    base_url=base_url
)
client

In [14]:
def embed_texts_batched(
    texts,
    model="text-embedding-3-small",
    max_batch_tokens=200000,
    max_batch_items=64
):
    all_embeddings = []

    batches = make_batches(
        texts,
        max_batch_tokens=max_batch_tokens,
        max_batch_items=max_batch_items
    )

    print(f"Total texts: {len(texts)}")
    print(f"Total batches: {len(batches)}")

    processed = 0
    for batch_idx, batch in enumerate(batches, start=1):
        response = client.embeddings.create(
            model=model,
            input=batch
        )

        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)

        processed += len(batch)
        print(f"Batch {batch_idx}/{len(batches)} done. Processed {processed}/{len(texts)}")

    return np.array(all_embeddings, dtype=np.float32)

In [16]:
article_embeddings = embed_texts_batched(
    chunks_df["retrieval_text"].tolist(),
    model=model,
    max_batch_tokens=200000,
    max_batch_items=64
)

print(article_embeddings.shape)

In [3]:
np.save("../data/chunk_embeddings_2025.npy", article_embeddings)
# chunks_df.to_parquet("../data/chunk_metadata_2025.parquet", index=False, engine='pyarrow')
chunks_df.to_csv("../data/chunk_metadata.csv", index=False)

Error: 

In [25]:
len(chunks_df) == len(article_embeddings)